# RAG 是什么？
RAG = Retrieval-Augmented Generation（检索增强生成）

简单说就是：让AI在回答问题之前，先去查资料，然后再回答。

# 文本分割：
## 文本分割主要考虑两个因素：
1）embedding模型的Tokens限制情况；2）语义完整性对整体的检索效果的影响。一些常见的文本分割方式如下：

句分割：以”句”的粒度进行切分，保留一个句子的完整语义。常见切分符包括：句号、感叹号、问号、换行符等。
固定长度分割：根据embedding模型的token长度限制，将文本分割为固定长度（如1024/512个tokens），这种切分方式会损失很多语义信息，一般通过在头尾增加一定冗余量来缓解。

In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 加载文档
loader = TextLoader("crossover_epic_saga.txt", encoding="utf-8")
documents = loader.load()

# 2. 切分文档
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # 每段500字
    chunk_overlap=50,    # 重叠50字，保持连贯
    length_function=len,
)

chunks = text_splitter.split_documents(documents)

print(f"原文档: {len(documents)} 个")
print(f"切分后: {len(chunks)} 段")
print(f"\n第一段内容:\n{chunks[0].page_content}")
print(f"\n第二段内容:\n{chunks[1].page_content}")

原文档: 1 个
切分后: 27 段

第一段内容:
# 次元裂缝：洛洛的终极冒险

## 第一章：异变降临

"机车战士们，准备出击！"

洛洛站在时光之城的高塔上，望着远处黑压压的猛兽族大军，握紧了拳头。自从来到机战王的世界，他已经带领机车族经历了无数次战斗，但今天的敌人似乎有些不同。

天空突然裂开了一道巨大的缝隙，紫色的闪电在裂缝中游走。那不是普通的闪电，而是某种更加诡异的力量。

"洛洛！那是什么？"霹雳火仰头望着天空，红色的装甲在闪电的映照下显得格外醒目。

洛洛还没来得及回答，裂缝中突然射出一道金光，将他和霹雳火同时笼罩。在失去意识前的最后一刻，洛洛听到了一个机械般的声音：

"检测到符合条件的'王'之资质，启动次元召唤程序……"

---

当洛洛再次睁开眼睛时，他发现自己躺在一片陌生的沙滩上。碧蓝的大海一望无际，远处有几艘帆船正在航行。最让他震惊的是，霹雳火就躺在他身边，但体型变小了——不再是那台巨大的机车战士，而是变成了只有一人高的机器人形态。

"霹雳火！你没事吧？"

"洛洛……我感觉很奇怪，"霹雳火坐起身，看着自己的双手，"我的力量……变弱了？"

第二段内容:
"洛洛……我感觉很奇怪，"霹雳火坐起身，看着自己的双手，"我的力量……变弱了？"

洛洛检查了一下自己的状态，发现机战王的操控系统还在，但只能感应到霹雳火一个单位。其他机车战士——力霸天、冲击波、龙卷风……全都失去了联系。

"看来我们被传送到了另一个世界，"洛洛分析道，"而且这里的规则不同，你们的体型和力量都受到了限制。"

就在这时，沙滩另一头传来一阵喧闹声。洛洛和霹雳火警惕地望去，只见一个戴着草帽的少年正被一群海盗模样的人追赶。

"把财宝交出来，小子！"

"我才不要呢！这是我找到的！"草帽少年灵活地躲避着攻击，脸上挂着灿烂的笑容，"我可是要成为海贼王的男人！"

洛洛瞳孔一缩。海贼王？他当然知道这个著名的漫画世界。但为什么他会来到这里？那道裂缝到底是什么？

"需要帮忙吗？"洛洛走上前问道。

草帽少年——路飞愣了一下，然后大笑起来："哈哈哈！不用不用，这种小角色我自己就能搞定！橡胶橡胶——手枪！"

他的手臂突然伸长，一拳将追在最前面的海盗打飞了出去。

洛洛和霹雳火对视一眼，都从对方眼中看到了震惊。这个世界的能力体系……完全不同于机战王的世界！

---


# 向量化（embedding）：
向量化是一个将文本数据转化为向量矩阵的过程，该过程会直接影响到后续检索的效果。目前常见的embedding模型基本能满足大部分需求，但对于特殊场景（例如涉及一些罕见专有词或字等）或者想进一步优化效果，则可以选择开源Embedding模型微调或直接训练适合自己场景的Embedding模型。这里嵌入模型使用Qwen3-Embedding-0.6B

In [6]:
from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn
import numpy as np

# 模型路径
model_path = r"C:\Users\吴创捷\Desktop\生产\agent开发\qwen3-embedding-model\Qwen3-Embedding-0.6B\qwen\Qwen3-Embedding-0.6B"

print("🚀 加载模型中...")
print(f"路径: {model_path}")

# 用 SentenceTransformer 加载（Transformer → Pooling → Normalize 流水线）
# trust_remote_code 去掉，模型无 auto_map 不需要
model = SentenceTransformer(model_path)

print("✅ 模型加载成功！")
print(f"模型类型: {type(model).__name__}")

# 测试文本
texts = [
    "洛洛是机战王",
    "路飞是要成为海贼王的男人",
    "孙悟空会龟派气功",
    "今天天气真好"
]

print(f"\n📝 测试 {len(texts)} 个句子:\n")

# 批量编码
embeddings = model.encode(texts, normalize_embeddings=True)

for text, emb in zip(texts, embeddings):
    print(f"文本: {text}")
    print(f"向量维度: {emb.shape}")
    print(f"向量前5个值: {emb[:5]}")
    print()

# 测试语义相似度
print("🔍 测试语义相似度:")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

text1 = "洛洛是机战王"
text2 = "洛洛操控机车战士"
text3 = "今天吃了苹果"

emb1 = model.encode(text1, normalize_embeddings=True)
emb2 = model.encode(text2, normalize_embeddings=True)
emb3 = model.encode(text3, normalize_embeddings=True)

print(f"'{text1}' vs '{text2}': {cosine_similarity(emb1, emb2):.4f}")
print(f"'{text1}' vs '{text3}': {cosine_similarity(emb1, emb3):.4f}")
print(f"\n相似度越高表示语义越接近！")

🚀 加载模型中...
路径: C:\Users\吴创捷\Desktop\生产\agent开发\qwen3-embedding-model\Qwen3-Embedding-0.6B\qwen\Qwen3-Embedding-0.6B


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

✅ 模型加载成功！
模型类型: SentenceTransformer

📝 测试 4 个句子:

文本: 洛洛是机战王
向量维度: (1024,)
向量前5个值: [ 0.05932617 -0.01318359 -0.00567627 -0.04858398  0.00634766]

文本: 路飞是要成为海贼王的男人
向量维度: (1024,)
向量前5个值: [ 0.08203125  0.02832031 -0.00860596 -0.03881836  0.01098633]

文本: 孙悟空会龟派气功
向量维度: (1024,)
向量前5个值: [-0.0062561  -0.05126953 -0.00848389 -0.0078125   0.0559082 ]

文本: 今天天气真好
向量维度: (1024,)
向量前5个值: [-0.02062988 -0.01464844 -0.00817871 -0.0112915   0.05859375]

🔍 测试语义相似度:
'洛洛是机战王' vs '洛洛操控机车战士': 0.7296
'洛洛是机战王' vs '今天吃了苹果': 0.2351

相似度越高表示语义越接近！


# 向量存储（Vector Store）

向量化完成后，需要把所有文本块（chunks）转成向量存入数据库。这样用户提问时，只需把问题向量化，在库中快速搜索最相关的 Top-K 段落，无需每次重新编码全部文档。

## 为什么需要向量数据库？

- **高效检索**：向量库专门优化了近似最近邻搜索（ANN），毫秒级从海量向量中找到最相似的 Top-K
- **持久化存储**：向量存到磁盘，重启后直接加载，不用重新编码
- **增量维护**：支持添加、删除、更新向量

## 常见选择

| 工具 | 特点 |
|------|------|
| **Chroma** | 轻量、易用，LangChain 原生集成，适合学习和中小规模项目 |
| **FAISS** | Meta 开源，性能极高，适合大规模检索 |
| **Milvus** | 分布式向量数据库，适合生产环境 |

这里使用 Chroma——最简单，几行代码就能跑起来。

In [7]:
# 如果还没安装 Chroma，先运行：pip install chromadb

import chromadb
from chromadb.config import Settings

# 1. 初始化 Chroma 客户端（数据持久化到本地磁盘）
client = chromadb.PersistentClient(path="./chroma_db")

# 2. 创建或获取一个 collection（类似关系数据库中的"表"）
#    如果已存在同名 collection 就先删掉重建，保证每次运行结果一致
if "crossover_story" in [c.name for c in client.list_collections()]:
    client.delete_collection("crossover_story")

collection = client.create_collection(
    name="crossover_story",
    metadata={"description": "次元裂缝：洛洛的终极冒险 文本块"}
)

# 3. 把之前切好的 27 个 chunks 逐条编码并存入
#    每条记录需要：id（唯一标识）、document（原文）、embeddings（向量，可选）
texts = [chunk.page_content for chunk in chunks]  # 提取文本内容
ids = [f"chunk_{i:03d}" for i in range(len(chunks))]  # 生成唯一 ID

# 用之前加载的 Qwen3-Embedding 模型批量编码
embeddings = model.encode(texts, normalize_embeddings=True)

# 存入 collection
collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist()  # numpy → list
)

print(f"✅ 已存储 {collection.count()} 个文本块到向量数据库")
print(f"存储路径: ./chroma_db/")
print(f"向量维度: {embeddings.shape[1]}")

# 4. 测试检索 —— 问一个问题，看能否找回相关段落
query = "洛洛是怎么穿越到海贼王世界的？"
query_embedding = model.encode([query], normalize_embeddings=True)

results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3  # 返回最相关的 3 段
)

print(f"\n🔍 查询: {query}")
print(f"返回 {len(results['documents'][0])} 个最相关的文本块:\n")
for i, (doc_id, doc_text, distance) in enumerate(zip(
    results["ids"][0], results["documents"][0], results["distances"][0]
), 1):
    print(f"--- 第{i}段 | ID: {doc_id} | 距离: {distance:.4f} ---")
    print(doc_text[:200])
    print()

✅ 已存储 27 个文本块到向量数据库
存储路径: ./chroma_db/
向量维度: 1024

🔍 查询: 洛洛是怎么穿越到海贼王世界的？
返回 3 个最相关的文本块:

--- 第1段 | ID: chunk_001 | 距离: 0.7169 ---
"洛洛……我感觉很奇怪，"霹雳火坐起身，看着自己的双手，"我的力量……变弱了？"

洛洛检查了一下自己的状态，发现机战王的操控系统还在，但只能感应到霹雳火一个单位。其他机车战士——力霸天、冲击波、龙卷风……全都失去了联系。

"看来我们被传送到了另一个世界，"洛洛分析道，"而且这里的规则不同，你们的体型和力量都受到了限制。"

就在这时，沙滩另一头传来一阵喧闹声。洛洛和霹雳火警惕地望去，只见一个戴

--- 第2段 | ID: chunk_002 | 距离: 0.7362 ---
---

## 第二章：命运的交汇

路飞三拳两脚就解决了那群海盗，然后好奇地围着霹雳火转圈。

"哇！你是机器人吗？好酷！会变形吗？"

霹雳火有些无奈："我是机车族的战士，平时是跑车的形态，但现在……"

"现在我们的力量被这个世界压制了，"洛洛解释道，"我叫洛洛，是机战王。这位是霹雳火，我的伙伴。"

"机战王？那是什么？能吃吗？"路飞挠挠头。

洛洛哭笑不得："不是吃的……简单来说，我是能

--- 第3段 | ID: chunk_025 | 距离: 0.8008 ---
洛洛笑了："那就约定好了！无论身在哪个世界，我们都是伙伴！"

"伙伴！"众人齐声喊道。

哆啦A梦拿出任意门："我送你们回去吧。"

洛洛和机车战士们走进任意门，在跨入门的那一刻，他回头望了一眼这片大海。

"路飞……等着我。总有一天，我会找到你，然后一起开宴会！"

门缓缓关闭，洛洛回到了机战王的世界。

---

时光之城，阳光明媚。

洛洛站在高塔上，望着远处的风景。猛兽族已经撤退，机车族



# 检索（Retrieval）

向量库建好后，RAG 的 "R"（Retrieval）登场——根据用户问题，从向量库中找出最相关的文本块作为参考资料。

## 检索流程

1. 把用户问题用同一个 Embedding 模型向量化
2. 在向量库中做相似度搜索
3. 返回 Top-K 个最相关的文本块

## 关键参数

- **Top-K**：返回结果数量。太少可能遗漏关键信息，太多会引入噪声干扰 LLM 判断
- **检索方式**：
  - **语义检索**（默认）：向量相似度匹配，能理解"穿越"和"传送"是近义词，但不一定精准命中专有名词
  - **关键词检索（BM25）**：传统关键词匹配，适合专有名词和精确查找
  - **混合检索**：两者结合，取长补短，生产环境常用

这里先用语义检索，看看效果如何。

In [9]:
def retrieve(query, top_k=3):
    """将问题向量化，在 Chroma 中检索最相关的 top_k 个文本块"""
    query_embedding = model.encode([query], normalize_embeddings=True)
    return collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
    )


def format_retrieved(results):
    """把检索结果整理成 Prompt 可用的文本格式"""
    lines = []
    for i, (doc_id, doc_text, distance) in enumerate(zip(
        results["ids"][0], results["documents"][0], results["distances"][0]
    ), 1):
        lines.append(f"[参考资料 {i}] (来源: {doc_id}, 距离: {distance:.4f})")
        lines.append(doc_text)
        lines.append("")
    return "\n".join(lines)


# ─── 测试检索效果 ───

test_queries = [
    "洛洛是怎么穿越到海贼王世界的？",
    "霹雳火穿越后有什么变化？",
    "路飞用了什么招式？",
]

for query in test_queries:
    results = retrieve(query, top_k=2)
    print(f"🔍 问题: {query}")
    for i, (doc_id, doc_text, distance) in enumerate(zip(
        results["ids"][0], results["documents"][0], results["distances"][0]
    ), 1):
        print(f"  Top-{i} | {doc_id} | 距离: {distance:.4f}")
        print(f"  {doc_text[:120]}...")
    print()

🔍 问题: 洛洛是怎么穿越到海贼王世界的？
  Top-1 | chunk_001 | 距离: 0.7169
  "洛洛……我感觉很奇怪，"霹雳火坐起身，看着自己的双手，"我的力量……变弱了？"

洛洛检查了一下自己的状态，发现机战王的操控系统还在，但只能感应到霹雳火一个单位。其他机车战士——力霸天、冲击波、龙卷风……全都失去了联系。

"看来我们被传...
  Top-2 | chunk_002 | 距离: 0.7362
  ---

## 第二章：命运的交汇

路飞三拳两脚就解决了那群海盗，然后好奇地围着霹雳火转圈。

"哇！你是机器人吗？好酷！会变形吗？"

霹雳火有些无奈："我是机车族的战士，平时是跑车的形态，但现在……"

"现在我们的力量被这个世界压制...

🔍 问题: 霹雳火穿越后有什么变化？
  Top-1 | chunk_017 | 距离: 0.9198
  "雷霆半月斩！"

霹雳火的剑气斩向弗利萨，迫使他收回攻击躲避。

"又是你，机器人！"弗利萨眼中闪过一丝不耐，"先解决你！"

他双手连挥，数十道死亡光束射向霹雳火。霹雳火运用气的屏障抵挡，但光束的威力太强，屏障很快出现了裂痕。

"力霸...
  Top-2 | chunk_001 | 距离: 0.9306
  "洛洛……我感觉很奇怪，"霹雳火坐起身，看着自己的双手，"我的力量……变弱了？"

洛洛检查了一下自己的状态，发现机战王的操控系统还在，但只能感应到霹雳火一个单位。其他机车战士——力霸天、冲击波、龙卷风……全都失去了联系。

"看来我们被传...

🔍 问题: 路飞用了什么招式？
  Top-1 | chunk_017 | 距离: 0.7633
  "雷霆半月斩！"

霹雳火的剑气斩向弗利萨，迫使他收回攻击躲避。

"又是你，机器人！"弗利萨眼中闪过一丝不耐，"先解决你！"

他双手连挥，数十道死亡光束射向霹雳火。霹雳火运用气的屏障抵挡，但光束的威力太强，屏障很快出现了裂痕。

"力霸...
  Top-2 | chunk_010 | 距离: 0.8724
  "三角冲击波！"

三道能量光束汇聚成一点，直接轰向弗利萨的战舰！

"什么？！"弗利萨不得不分心抵挡，给了孙悟空可乘之机。

"龟派气功！"

蓝色的能量波贯穿天空，正中弗利萨的胸口！

"可恶……"弗利萨被轰退了数百米

# 增强生成（Generation）

RAG 的最后一步 "G"（Generation）——把检索到的资料和用户问题一起交给 LLM，生成有据可查的答案。

## 工作流程

```
用户问题 → 向量检索 → Top-K 文本块 → 拼入 Prompt → LLM 生成答案
```

## 一个好的 RAG Prompt 包含什么？

1. **系统角色**：告诉 LLM 它的职责（如"你是一个严谨的知识问答助手"）
2. **参考资料**：检索到的文本块（这是 RAG 的核心价值）
3. **用户问题**：用户的原始提问
4. **约束条件**："仅根据资料回答"、"不知道就说不知道"——这一步很关键，能有效减少模型胡说八道（幻觉）</cell id="1efc76388d2062c6">


In [11]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

# 初始化 LLM（使用 OpenAI 兼容接口，本地/远程模型均可）
# 模型名、Key、URL 均从 .env 文件读取
llm = ChatOpenAI(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "enabled"}}
)


def rag_answer(query, top_k=3):
    """完整的 RAG 流程：检索 + 生成"""

    # R 步：检索相关资料
    results = retrieve(query, top_k=top_k)
    context = format_retrieved(results)

    # G 步：构建 Prompt + 调用 LLM 生成
    system_prompt = (
        "你是一个严谨的知识问答助手。"
        "请严格根据下面给出的参考资料回答用户问题。"
        "如果参考资料中没有足够信息，直接说'根据现有资料无法回答'，绝不编造。"
    )

    user_prompt = (
        f"参考资料：\n{context}\n"
        f"用户问题：{query}\n"
        f"请根据以上资料回答问题："
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    response = llm.invoke(messages)
    return response.content, context


# ─── 测试完整 RAG 流程 ───

question = "洛洛是怎么穿越到海贼王世界的？"
answer, sources = rag_answer(question)

print("=" * 60)
print(f"❓ 用户问题: {question}")
print("=" * 60)
print(f"\n📖 检索到的参考资料:\n{sources}")
print(f"🤖 AI 回答:\n{answer}")
print("=" * 60)

# 再测一个
question2 = "霹雳火穿越后有什么变化？"
answer2, sources2 = rag_answer(question2)

print(f"\n\n❓ 用户问题: {question2}")
print("=" * 60)
print(f"🤖 AI 回答:\n{answer2}")

❓ 用户问题: 洛洛是怎么穿越到海贼王世界的？

📖 检索到的参考资料:
[参考资料 1] (来源: chunk_001, 距离: 0.7169)
"洛洛……我感觉很奇怪，"霹雳火坐起身，看着自己的双手，"我的力量……变弱了？"

洛洛检查了一下自己的状态，发现机战王的操控系统还在，但只能感应到霹雳火一个单位。其他机车战士——力霸天、冲击波、龙卷风……全都失去了联系。

"看来我们被传送到了另一个世界，"洛洛分析道，"而且这里的规则不同，你们的体型和力量都受到了限制。"

就在这时，沙滩另一头传来一阵喧闹声。洛洛和霹雳火警惕地望去，只见一个戴着草帽的少年正被一群海盗模样的人追赶。

"把财宝交出来，小子！"

"我才不要呢！这是我找到的！"草帽少年灵活地躲避着攻击，脸上挂着灿烂的笑容，"我可是要成为海贼王的男人！"

洛洛瞳孔一缩。海贼王？他当然知道这个著名的漫画世界。但为什么他会来到这里？那道裂缝到底是什么？

"需要帮忙吗？"洛洛走上前问道。

草帽少年——路飞愣了一下，然后大笑起来："哈哈哈！不用不用，这种小角色我自己就能搞定！橡胶橡胶——手枪！"

他的手臂突然伸长，一拳将追在最前面的海盗打飞了出去。

洛洛和霹雳火对视一眼，都从对方眼中看到了震惊。这个世界的能力体系……完全不同于机战王的世界！

---

[参考资料 2] (来源: chunk_002, 距离: 0.7362)
---

## 第二章：命运的交汇

路飞三拳两脚就解决了那群海盗，然后好奇地围着霹雳火转圈。

"哇！你是机器人吗？好酷！会变形吗？"

霹雳火有些无奈："我是机车族的战士，平时是跑车的形态，但现在……"

"现在我们的力量被这个世界压制了，"洛洛解释道，"我叫洛洛，是机战王。这位是霹雳火，我的伙伴。"

"机战王？那是什么？能吃吗？"路飞挠挠头。

洛洛哭笑不得："不是吃的……简单来说，我是能操控机车战士战斗的人。"

"哦！"路飞眼睛一亮，"那你能操控多少机器人？"

"在我的世界，我能操控所有机车族的战士。但在这里……"洛洛尝试感应其他机车战士的存在，却一无所获，"只能感应到霹雳火。"

"听起来很厉害的样子！"路飞兴奋地跳起来，"我叫蒙奇·D·路飞！是要成为海贼王的男人！你们跟我一起走吧！"

"等等，"洛洛拦住他，"你知道那道天空中的裂缝是怎么回事吗？"

路飞抬头看了